# Amherst Temperature Demo
This test uses daily average temperatures and asks LLB to infer latent temperature, latent volatility, and forecast the next 2 days.

In [ ]:
import io
import os
import re
from contextlib import redirect_stdout

import llb
import numpy as np

# Informal user text: predict only the next day.
text = (
    "I have daily average temperature measurements in Amherst, Massachusetts. "
    "Temperature evolves smoothly over time, but observations are noisy and weather conditions can vary in stability. "
    "I want to model the underlying true temperature and how uncertainty changes over time, "
    "and predict the average temp of the next day."
)

# DATA
# Daily average temperatures (F) for Mar 1-19.
avg_temperature = [
    31.6, 20.4, 23.9, 33.9, 34.1, 34.0, 37.5, 42.9, 38.7,
    52.4, 55.4, 49.2, 34.8, 37.4, 36.2, 50.2, 42.0, 28.3, 33.1
]
num_days = len(avg_temperature)

data = {
    "num_days": num_days,
    "average_temperature": avg_temperature,
}

targets = ["latent_temperature", "latent_volatility", "future_average_temperature"]

# Run infer once, keep weighted posterior, and parse flat summary from infer logs.
buf = io.StringIO()
with redirect_stdout(buf):
    posterior = llb.infer(
        text=text,
        data=data,
        targets=targets,
        api_url="https://api.openai.com/v1/chat/completions",
        api_key="OPENAI_API_KEY",
        api_model="gpt-4.1-mini",
        n_models=50,
        mcmc_num_warmup=200,
        mcmc_num_samples=800,
        random_seed=42,
    )
infer_log = buf.getvalue()
print(infer_log, end="")

print("Returned keys:", list(posterior.keys()))

latent_temp = np.asarray(posterior["latent_temperature"], dtype=float)
latent_vol = np.asarray(posterior["latent_volatility"], dtype=float)
future_temp = np.asarray(posterior["future_average_temperature"], dtype=float)

latent_temp_mean = latent_temp.mean(axis=0)
latent_vol_mean = latent_vol.mean(axis=0)

# Weighted next-day prediction from returned posterior.
future_temp_mean = np.asarray(future_temp.mean(axis=0), dtype=float).reshape(-1)
pred_w_next_day = float(future_temp_mean[0])

# Flat next-day prediction parsed from infer's weighted-vs-flat summary.
pred_f_next_day = None
block_pattern = r"--- Weighted vs Flat Target Summary ---\nTarget: future_average_temperature\n([\s\S]*?)(?:\n---|\Z)"
m = re.search(block_pattern, infer_log)
if m:
    block = m.group(1)
    m_scalar = re.search(r"flat mean prediction:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", block)
    if m_scalar:
        pred_f_next_day = float(m_scalar.group(1))
    else:
        m_vec = re.search(r"flat mean prediction (?:full|first_10):\s*\[([^\]]+)\]", block)
        if m_vec:
            nums = [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", m_vec.group(1))]
            if nums:
                pred_f_next_day = float(nums[0])

if pred_f_next_day is None:
    raise RuntimeError("Could not parse flat mean prediction for future_average_temperature from infer output.")

print("latent_temperature mean first_10:", latent_temp_mean[:10].tolist())
print("latent_volatility mean first_10:", latent_vol_mean[:10].tolist())
print("future_average_temperature weighted mean (day 1):", pred_w_next_day)
print("future_average_temperature flat mean (day 1):", pred_f_next_day)
print("difference (weighted - flat):", pred_w_next_day - pred_f_next_day)

# Real next day you provided (Mar 20 average temperature).
actual_next_day = 42.7
err_w = pred_w_next_day - actual_next_day
err_f = pred_f_next_day - actual_next_day
abs_err_w = abs(err_w)
abs_err_f = abs(err_f)
pct_err_w = (abs_err_w / actual_next_day) * 100.0
pct_err_f = (abs_err_f / actual_next_day) * 100.0

print("--- Compare With Real Next Day (Mar 20) ---")
print("actual next day:", actual_next_day)
print("weighted predicted next day:", pred_w_next_day)
print("weighted error (pred-actual):", err_w)
print("weighted absolute error:", abs_err_w)
print("weighted percent error:", pct_err_w)
print("flat predicted next day:", pred_f_next_day)
print("flat error (pred-actual):", err_f)
print("flat absolute error:", abs_err_f)
print("flat percent error:", pct_err_f)

winner = "weighted" if abs_err_w < abs_err_f else ("flat" if abs_err_f < abs_err_w else "tie")
print(f"Winner vs actual next day: {winner}")

/Users/hoanghiepg6/LLB - Professor Justin Domke/llb/mcmc_log.py:37: UserWarning: Missing a plate statement for batch dimension -3 at site 'obs_temperature'. You can use `numpyro.util.format_shapes` utility to check shapes at all sites of your model.
  mcmc.run(jax.random.PRNGKey(rng_seed), data=data)


Number of requested models: 50
Number of generated models: 50
Number of deduplicated models: 0
Number of invalid models (syntax/parsing): 0
Number of generation request failures (timeout/network/API): 0
Number of models missing required targets: 0
Number of models that failed to compile: 0
Number of models that failed during inference: 8
Number of models dropped due to target shape mismatch: 8
Number of models dropped due to non-finite log bound: 0
Number of valid models used in final aggregation: 34
--- Model Averaging Summary ---
Target: latent_temperature
Number of models: 34
flat mean prediction shape: (19,)
flat mean prediction first_10: [27.63256122267146, 23.163312639237024, 24.561482772531654, 29.931691437450745, 31.291971933555736, 32.14103681021141, 34.722913824734974, 38.03267736921458, 37.86801610205528, 45.48321209274785]
weighted mean prediction shape: (19,)
weighted mean prediction first_10: [26.361255434519727, 23.80030995752827, 26.292623360280906, 31.97518685931673, 3